# Bài 8
Đây là notebook chứa mã nguồn đầy đủ của bài 8.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
def solve_bai08(discount=0.05, capital_growth=0.06, target_ai=0.85, budget_growth=0.08):
    T = 10
    years = list(range(2026, 2036))

    K0  = 25900.0  
    D0  = 20.0
    AI0 = 0.60
    H0  = 30.0
    base_budget = 100.0

    budgets = [base_budget * (1 + budget_growth)**t for t in range(T)]

    def simulate(alloc_matrix, shock_2028=False):
        # alloc_matrix shape: (T, 4) -> K, D, AI, H
        K, D, AI, H = K0, D0, AI0, H0
        Y_series, C_series = [], []
        K_s, D_s, AI_s, H_s = [], [], [], []
        
        for t in range(T):
            if shock_2028 and years[t] == 2028:
                K *= 0.92  # Shock reduces capital -> reduces Y indirectly

            invest_K  = budgets[t] * alloc_matrix[t, 0]
            invest_D  = budgets[t] * alloc_matrix[t, 1]
            invest_AI = budgets[t] * alloc_matrix[t, 2]
            invest_H  = budgets[t] * alloc_matrix[t, 3]

            Y = (K / 1000)**0.4 * D**0.15 * AI**0.2 * H**0.25 * 300
            
            if shock_2028 and years[t] == 2028:
                Y *= 0.92  # Direct shock to Y
                
            C = Y * 0.65  # Consumption

            Y_series.append(Y)
            C_series.append(C)
            K_s.append(K)
            D_s.append(D)
            AI_s.append(AI)
            H_s.append(H)

            # Update for next year
            K  = K * (1 + capital_growth) + invest_K * 10
            D  = D + invest_D * 0.5
            AI = min(1.0, AI + 0.05 * invest_AI / 100)
            H  = H + invest_H * 0.2

        return np.array(K_s), np.array(D_s), np.array(AI_s), np.array(H_s), np.array(Y_series), np.array(C_series)

    def objective(x_flat, shock=False):
        alloc_matrix = x_flat.reshape((T, 4))
        _, _, AI_s, _, Y_series, _ = simulate(alloc_matrix, shock_2028=shock)
        discounted_welfare = sum(Y_series[t] / (1 + discount)**t for t in range(T))
        penalty = max(0, target_ai - AI_s[-1]) * 2000
        return -(discounted_welfare - penalty)

    def constraint_sum(x_flat):
        alloc_matrix = x_flat.reshape((T, 4))
        # sum of fractions each year must be 1.0
        return 1.0 - np.sum(alloc_matrix, axis=1)

    x0 = np.full(4 * T, 0.25)
    bounds = [(0.05, 0.8)] * (4 * T)
    cons = {'type': 'eq', 'fun': constraint_sum}

    # 1. Base optimization (SLSQP)
    res = minimize(objective, x0, args=(False,), bounds=bounds, constraints=cons, method='SLSQP', options={'maxiter': 200})
    opt_alloc = res.x.reshape((T, 4))
    K_opt, D_opt, AI_opt, H_opt, Y_opt, C_opt = simulate(opt_alloc)
    opt_welfare = -res.fun

    # 3. Shock analysis
    res_shock = minimize(objective, x0, args=(True,), bounds=bounds, constraints=cons, method='SLSQP', options={'maxiter': 200})
    shock_alloc = res_shock.x.reshape((T, 4))
    K_sh, D_sh, AI_sh, H_sh, Y_sh, C_sh = simulate(shock_alloc, shock_2028=True)
    shock_welfare = -res_shock.fun

    # 4. Strategies comparison
    # (i) Even
    alloc_even = np.full((T, 4), 0.25)
    _, _, _, _, Y_even, _ = simulate(alloc_even)
    welfare_even = sum(Y_even[t] / (1 + discount)**t for t in range(T))
    
    # (ii) Front-load (more investment in first 3 years, means budget multiplier changes)
    # We will simulate front load by shifting budget weights
    budget_front = budgets.copy()
    total_b = sum(budgets)
    front_ratio = [0.15, 0.15, 0.15] + [0.55/7]*7
    budget_front = [total_b * r for r in front_ratio]
    
    def simulate_custom_budget(budgets_arr):
        K, D, AI, H = K0, D0, AI0, H0
        Y_series = []
        for t in range(T):
            invest_K = budgets_arr[t] * 0.25
            invest_D = budgets_arr[t] * 0.25
            invest_AI = budgets_arr[t] * 0.25
            invest_H = budgets_arr[t] * 0.25
            Y = (K / 1000)**0.4 * D**0.15 * AI**0.2 * H**0.25 * 300
            Y_series.append(Y)
            K  = K * (1 + capital_growth) + invest_K * 10
            D  = D + invest_D * 0.5
            AI = min(1.0, AI + 0.05 * invest_AI / 100)
            H  = H + invest_H * 0.2
        return Y_series

    Y_front = simulate_custom_budget(budget_front)
    welfare_front = sum(Y_front[t] / (1 + discount)**t for t in range(T))

    return {
        'years': years,
        # Base
        'K': K_opt.tolist(),
        'D': D_opt.tolist(),
        'AI': AI_opt.tolist(),
        'H': H_opt.tolist(),
        'Y': Y_opt.tolist(),
        'C': C_opt.tolist(),
        'welfare_opt': opt_welfare,
        
        # Shock
        'Y_shock': Y_sh.tolist(),
        'welfare_shock': shock_welfare,
        
        # Strategies
        'welfare_even': welfare_even,
        'welfare_front': welfare_front,
        'better_strategy': 'Front-load' if welfare_front > welfare_even else 'Even',
    }

In [ ]:
if __name__ == '__main__':
    res = solve_bai08()
    # In ra một số key để kiểm tra
    if isinstance(res, dict):
        print(res.keys())